# Activity: Should We Trust the Alarm? — FreshKart Cold-Chain Case

**Unit 4 · AI-Assisted Visualization · AI-Assisted Pattern and Anomaly Visualization**
Visualization of Data, Understanding and Interpretation (UE24CS342AA9)

---

## The case

FreshKart's dark stores keep frozen food in **8 freezers** (one per store in Bengaluru, Pune and Hyderabad). Each freezer reports its temperature every hour. Last month FreshKart bought **ColdSense**, an AI service that scores every reading for "anomaly" and raises an alert when the score is **≥ 0.5**.

Cold-chain lead **Ravi** has six weeks of readings, the ColdSense scores, and the **maintenance log** (what technicians actually found). His technicians are tired of being sent out at night. He asks:

> *"Is ColdSense's alert feed trustworthy? Which alerts should my team chase, and what is it missing?"*

**You are the analyst.** Use the data and the visualizations to verify, correct or reject what the vendor claims.

**Ground rules (from the lecture):** an AI score is a *derived attribute*: look at it, don't only threshold it · flags are **candidates, not conclusions** · report precision and recall, not accuracy · scale emphasis to confidence.

**How to work:** run the notebook top to bottom. Complete every cell marked `# TODO` (replace `None` with your code or answer) and write in the cells marked **✏️ Your answer**. Cells that depend on your TODO will tell you if something is still missing.

### The data: `freshkart_coldchain.csv` (synthetic)

| Column | Meaning |
|---|---|
| `timestamp` | hour of the reading (1 Aug – 11 Sep 2026) |
| `city`, `store_id`, `freezer_id` | where the freezer is |
| `temp_c` | freezer temperature (°C); normal is about −18 °C |
| `door_opens` | number of door openings in that hour (busy by day, quiet at night) |
| `compressor_kw` | compressor power draw (kW) |
| `ai_score`, `ai_flag` | ColdSense anomaly score (0–1) and alert (1 if score ≥ 0.5) |
| `confirmed_fault`, `fault_type` | from the maintenance log: 1 if technicians confirmed a real fault in that hour, and what it was |

**Domain note from the maintenance team:** every freezer runs an automatic **defrost cycle from 02:00 to 04:00** each night. The heater warms the coil, so the temperature rises for those two hours. That is normal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (6, 3.2), "axes.spines.top": False, "axes.spines.right": False})
GREY, ACCENT, BLUE = "#bdbdbd", "#d95f02", "#4c78a8"

def ready(*vals):
    """Helper: tells you if a TODO above is still unanswered."""
    ok = all(v is not None for v in vals)
    if not ok:
        print("⚠ Complete the TODO cell(s) above first, then re-run this cell.")
    return ok

df = pd.read_csv("freshkart_coldchain.csv", parse_dates=["timestamp"])
df["hour"] = df["timestamp"].dt.hour
df.head()

## ColdSense's monthly report (to be validated)

| # | Vendor statement |
|---|---|
| 1 | "ColdSense is **97.9% accurate**, so its alerts can be trusted." |
| 2 | "**Most faults happen around 2 am.** We recommend a night technician." |
| 3 | "**All 10 compressor trips** in the period were caught." |
| 4 | "Freezer **FZ-H2 ran without a single fault** in August." |
| 5 | "A reading of **−14 °C is abnormal** for any freezer, at any time." |
| 6 | "An alert with a score of **0.92 means a 92% chance** of a real fault." |
| 7 | "The **default threshold of 0.5** is the best setting for FreshKart." |
| 8 | "A new FZ-B1 reading of **−15.9 °C at 14:00** is **15 standard errors** above normal: a definite fault." |

---
## Part 1 · Data abstraction: items, attributes and derived attributes

Munzner: before choosing a view, ask what the **items** and **attributes** are. The AI does not change the items; it **derives** new attributes.

In [ ]:
derived_cols = ["ai_score", "ai_flag"]  # model-derived score and thresholded alert flag
n_alerts = int((df["ai_flag"] == 1).sum())
fault_rate_pct = round(df["confirmed_fault"].mean() * 100, 2)

if ready(derived_cols, n_alerts, fault_rate_pct):
    print(f"derived: {derived_cols} | alerts: {n_alerts} | faults: {df.confirmed_fault.sum()} of {len(df)} rows ({fault_rate_pct}%)")
    display(df[df.confirmed_fault == 1].groupby("fault_type").size().rename("fault hours"))

**✏️ Your answer (Part 1):** `ai_score` is quantitative and `ai_flag` is binary. Which **visual channel** suits each, and what information does the flag throw away that the score keeps?

Use **position or length** (for example, a dot on a numeric axis) or a continuous colour scale for the quantitative `ai_score`, because the exact score and ordering matter. Use a distinct categorical visual channel such as shape or a clearly labelled colour for the binary `ai_flag`; thresholding throws away how far a reading is above or below the cut-off and makes a score of 0.51 look identical to 0.99.

---
## Part 2 · Distributions: bin width and the score distribution

### 2a · Temperature histogram

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 5))
for col, w in enumerate([0.1, 1, 8]):
    for row, top in enumerate([None, 15]):
        ax = axes[row, col]
        ax.hist(df["temp_c"], bins=np.arange(-21, 0 + w, w), color=BLUE)
        if top: ax.set_ylim(0, top); ax.set_title(f"same, y-axis cut at {top} readings", fontsize=9)
        else: ax.set_title(f"bin width = {w} °C")
        ax.set_xlabel("temp (°C)")
plt.tight_layout(); plt.show()

# Fine bins preserve the warm tail as a distinct cluster; coarse bins can merge it.
warm_tail_visible_bins = [0.1, 1]
n_warm_readings = int((df["temp_c"] > -8).sum())

### 2b · Look at the AI scores before trusting the cut-off
Wilke: compare groups with overlapping densities, not stacked histograms.

In [ ]:
xs = np.linspace(0, 1, 300)
fig, ax = plt.subplots(figsize=(7, 3))
for lab, col in [(0, GREY), (1, ACCENT)]:
    s = df.loc[df.confirmed_fault == lab, "ai_score"]
    ax.fill_between(xs, stats.gaussian_kde(s)(xs), color=col, alpha=.5, label=f"{'real fault' if lab else 'normal'} (n={len(s)})")
ax.axvline(0.5, color="k", ls="--"); ax.set_yscale("log"); ax.set_ylim(1e-3, 20)
ax.set_xlabel("ai_score"); ax.legend(); ax.set_title("Score density, real faults vs normal readings (log y)")
plt.show()

scores_separate_faults = False

**✏️ Your answer (Part 2):** Why is a histogram "a parameterised algorithm", and what would you tell a colleague who says *"the 8 °C histogram shows no anomalies"*?

A histogram is a parameterised algorithm because the bin width, bin origin and axis limits determine how observations are grouped and which patterns become visible. The 8 °C view is too coarse to establish that no anomalies exist; I would inspect finer bins, the raw time series and the warm-tail readings before making that claim.

---
## Part 3 · Outlier rules on a short window (vendor statement 3)

Ten consecutive readings from **FZ-P1**, 12 Aug 06:00–15:00. One of them is a compressor trip.

In [ ]:
window = df[(df.freezer_id == "FZ-P1") & (df.timestamp >= "2026-08-12 06:00") & (df.timestamp < "2026-08-12 16:00")]
v = window["temp_c"].values
print(window[["timestamp", "temp_c", "fault_type"]].to_string(index=False))

z_max = np.max(np.abs((v - v.mean()) / np.std(v, ddof=1)))
q1, q3 = np.percentile(v, [25, 75])
upper_fence = q3 + 1.5 * (q3 - q1)

if ready(z_max, upper_fence):
    print(f"largest |z| = {z_max:.2f} -> flagged by |z| > 3? {z_max > 3}")
    print(f"upper IQR fence = {upper_fence:.2f} °C -> trip flagged by IQR rule? {v.max() > upper_fence}")
    print(f"largest |z| possible with n = {len(v)}: {(len(v) - 1) / np.sqrt(len(v)):.2f}")

**✏️ Your answer (Part 3):** Explain in two sentences why the z-score rule missed the trip here (use the word *masking*) and why the IQR rule did not.

The z-score rule missed the compressor trip because the extreme reading inflates the mean and standard deviation in this tiny window, causing **masking** and limiting the largest possible z-score. The IQR rule is more resistant to that extreme value, so the trip can lie beyond the upper IQR fence even when its z-score does not exceed 3.

---
## Part 4 · How good is ColdSense? (vendor statement 1)

In [ ]:
flag = df.ai_flag == 1
real = df.confirmed_fault == 1

TP = int((flag & real).sum())
FP = int((flag & ~real).sum())
FN = int((~flag & real).sum())
TN = int((~flag & ~real).sum())

if ready(TP, FP, FN, TN):
    display(pd.DataFrame([[TP, FN], [FP, TN]], index=["real fault", "normal"], columns=["alert", "no alert"]))

In [ ]:
precision = TP / (TP + FP) if TP + FP else 0
recall = TP / (TP + FN) if TP + FN else 0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
accuracy = (TP + TN) / len(df)
flag_nothing_accuracy = (df["confirmed_fault"] == 0).mean()

if ready(precision, recall, f1, accuracy, flag_nothing_accuracy):
    print(f"precision = {precision:.2f} | recall = {recall:.2f} | F1 = {f1:.2f} | accuracy = {accuracy:.1%} | never-alert accuracy = {flag_nothing_accuracy:.1%}")

**✏️ Your answer (Part 4):** Rewrite vendor statement 1 so it is honest. Use precision and recall, and say in plain words what they mean for Ravi's technicians (how many call-outs are wasted, how many faults are missed).

“At the current 0.5 threshold, ColdSense has the precision, recall and F1 values printed above; these are more informative than accuracy alone. Precision is the share of alerts that lead to a confirmed fault, so the remaining alerts create unnecessary call-outs; recall is the share of logged faults that are caught, so the remaining faults are missed.”

---
## Part 5 · Thresholds and costs (vendor statement 7)

A missed fault can spoil a freezer of stock (cost ≈ **40** units); a false call-out costs **1** unit of technician time.

In [ ]:
grid = np.round(np.arange(0.20, 0.96, 0.05), 2)
sweep = pd.DataFrame({"threshold": grid})
sweep["TP"] = [((df.ai_score >= g) & real).sum() for g in grid]
sweep["FP"] = [((df.ai_score >= g) & ~real).sum() for g in grid]
sweep["FN"] = real.sum() - sweep["TP"]
sweep["precision"] = (sweep.TP / (sweep.TP + sweep.FP)).round(2)
sweep["recall"] = (sweep.TP / real.sum()).round(2)

sweep["cost"] = 40 * sweep["FN"] + 1 * sweep["FP"]
best_threshold = float(sweep.loc[sweep["cost"].idxmin(), "threshold"])

if ready(best_threshold) and sweep["cost"].notna().all():
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.2))
    ax[0].plot(sweep.threshold, sweep.precision, color=BLUE, label="precision"); ax[0].plot(sweep.threshold, sweep.recall, color=ACCENT, label="recall")
    ax[0].axvline(0.5, color="k", ls="--", lw=.8); ax[0].legend(); ax[0].set_xlabel("threshold")
    ax[1].plot(sweep.threshold, sweep.cost.astype(float), color=BLUE, marker="o"); ax[1].axvline(best_threshold, color=ACCENT)
    ax[1].set_xlabel("threshold"); ax[1].set_ylabel("cost"); ax[1].set_title(f"lowest cost at {best_threshold}")
    plt.tight_layout(); plt.show()
    display(sweep)

**✏️ Your answer (Part 5):** Even at the best threshold, recall stays low. Why can no threshold fix this detector? (Hint: look at *which* faults are never scored high.)

A threshold only moves the cut-off on the scores the model already produces; it cannot recover faults that consistently receive low scores. If a fault type (or a collective sensor failure) is not represented by a high score, lowering the threshold may add many false alarms without reliably identifying that missing fault pattern; the detector needs better features or a different model, not just a new cut-off.

---
## Part 6 · Where do the errors come from?

### 6a · When do the false alarms happen? (vendor statement 2)

In [ ]:
fp_rows = df[flag & ~real]
fp_counts = fp_rows.groupby("hour").size().reindex(range(24), fill_value=0)
fp_counts.plot.bar(color=GREY, title="False alarms by hour of day", figsize=(9, 2.8))
plt.show()

fp_peak_hour = int(fp_counts.idxmax())
fp_share_at_peak = float(fp_counts.max() / len(fp_rows)) if len(fp_rows) else 0.0

if ready(fp_peak_hour, fp_share_at_peak):
    print(f"{fp_share_at_peak:.0%} of false alarms happen at {fp_peak_hour}:00")

### 6b · Which faults does ColdSense miss?

In [ ]:
caught = df[real].groupby("fault_type")["ai_flag"].agg(caught="sum", fault_hours="size")
caught["recall"] = (caught.caught / caught.fault_hours).round(2)
display(caught)

never_caught_type = caught.index[caught["caught"] == 0][0]
anomaly_kind = "collective"

### 6c · Contextual anomalies: compare each reading with its own hour

A **context-aware** baseline compares each reading with the same freezer at the same hour of day (median and MAD over the six weeks).

In [ ]:
grp = df.groupby(["freezer_id", "hour"])["temp_c"]
df["hour_median"] = grp.transform("median")
df["hour_mad"] = grp.transform(lambda s: 1.4826 * stats.median_abs_deviation(s))

df["z_context"] = (df["temp_c"] - df["hour_median"]) / df["hour_mad"].replace(0, np.nan)

if ready(*df["z_context"].head(1)):
    ctx_flag = df["z_context"].astype(float) > 4
    found = df[real & ctx_flag & ~flag]
    print(f"Real faults the context baseline catches but ColdSense missed: {len(found)}")
    display(found[["timestamp", "freezer_id", "temp_c", "hour_median", "z_context", "ai_score", "fault_type"]])
    print(f"False alarms at 02:00 under the context baseline: {(ctx_flag & ~real & (df.hour == 2)).sum()} (ColdSense: {(flag & ~real & (df.hour == 2)).sum()})")

### 6d · Collective anomalies: a run of readings that is abnormal together

In [ ]:
df = df.sort_values(["freezer_id", "timestamp"])
df["roll_sd_6h"] = df.groupby("freezer_id")["temp_c"].transform(lambda s: s.rolling(6).std())

flat = df[df["roll_sd_6h"] < 0.01].copy()

if ready(flat):
    print(flat.groupby("freezer_id")["timestamp"].agg(["min", "max", "count"]))
    h2 = df[(df.freezer_id == "FZ-H2") & (df.timestamp.between("2026-08-19 18:00", "2026-08-21 06:00"))]
    fig, ax = plt.subplots(figsize=(11, 2.8))
    ax.plot(h2.timestamp, h2.temp_c, color=GREY)
    # Highlight the full interval of near-constant readings, not isolated points.
    h2_flat = flat[(flat.freezer_id == "FZ-H2") & (flat.timestamp.between("2026-08-19 18:00", "2026-08-21 06:00"))]
    if not h2_flat.empty:
        ax.axvspan(h2_flat.timestamp.min(), h2_flat.timestamp.max(), color=ACCENT, alpha=.25)
    ax.set_title("FZ-H2: is anything unusual here?", loc="left"); plt.show()
df = df.sort_values(["timestamp", "store_id"])

**✏️ Your answer (Part 6):**
(a) Rewrite vendor statement 2. What really happens at 2 am, and what would happen if FreshKart hired a night technician based on the vendor's report?
(b) Is vendor statement 5 true? Use the words *contextual anomaly* and say what a chart must show for a viewer to judge such an alert.
(c) What is wrong with vendor statement 4, and why can a detector that scores readings one at a time never catch this kind of fault?

**(a)** “The 2 am peak is a peak in **false alarms**, not proof that most real faults happen at 2 am.” Hiring a night technician from this chart alone would likely send staff to investigate normal readings and would not necessarily improve fault detection; compare true faults by hour separately.

**(b)** Vendor statement 5 is not generally true: a temperature is a **contextual anomaly** only relative to the freezer's normal range at that hour. A useful chart must show the reading against that freezer's hour-specific baseline (median and spread), with the time, freezer identity and maintenance-log outcome visible.

**(c)** A flat run of readings from FZ-H2 can indicate a sensor that has frozen at one value, not a fault-free freezer. This is a **collective anomaly**: the unusual evidence is the sequence or run as a whole, so a detector that scores each reading independently may see every individual value as plausible and miss the pattern.

---
## Part 7 · Highlighting, saliency and frequency framing (vendor statement 6)

### 7a · Emphasis scaled to confidence
The vendor's dashboard paints every alert bright red. Implement the lecture's checklist: **≥ 0.90 strong**, **0.70–0.90 moderate**, **0.50–0.70 subtle outline**, **below 0.50 no emphasis** (the data stays visible).

In [ ]:
def emphasis(score):
    """Return (colour, marker size, edge colour) for one reading."""
    if score >= 0.90:
        return (ACCENT, 85, "black")
    elif score >= 0.70:
        return ("#e6a15a", 55, "black")
    elif score >= 0.50:
        return ("white", 32, ACCENT)
    else:
        return (GREY, 12, GREY)

week = df[(df.freezer_id == "FZ-B2") & (df.timestamp.between("2026-08-05", "2026-08-12"))]
if ready(emphasis(0.95)):
    fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
    ax[0].plot(week.timestamp, week.temp_c, color=GREY, lw=.8)
    a = week[week.ai_flag == 1]; ax[0].scatter(a.timestamp, a.temp_c, color="red", s=40); ax[0].set_title("Vendor view: every alert in red", loc="left")
    ax[1].plot(week.timestamp, week.temp_c, color=GREY, lw=.8)
    for _, r in week[week.ai_score >= 0.5].iterrows():
        col, size, edge = emphasis(r.ai_score)
        ax[1].scatter(r.timestamp, r.temp_c, color=col, s=size, edgecolors=edge)
    f_ = week[week.confirmed_fault == 1]; ax[1].scatter(f_.timestamp, f_.temp_c, marker="x", color="k", s=50, label="confirmed fault (log)")
    ax[1].legend(loc="upper left"); ax[1].set_title("Graded view", loc="left"); plt.tight_layout(); plt.show()

### 7b · "A score of 0.92 means 92%"? Count it.
Wilke: people read probabilities better as **frequencies**. Of all alerts in a score band, how many were real?

In [ ]:
band_hi = df[df.ai_score >= 0.9]
band_lo = df[(df.ai_score >= 0.5) & (df.ai_score < 0.7)]

real_per_10_hi = round(band_hi["confirmed_fault"].mean() * 10)
real_per_10_lo = round(band_lo["confirmed_fault"].mean() * 10)

if ready(real_per_10_hi, real_per_10_lo):
    fig, ax = plt.subplots(2, 1, figsize=(6, 1.8))
    for a_, k, lab in zip(ax, [real_per_10_hi, real_per_10_lo], ["score ≥ 0.9", "score 0.5–0.7"]):
        for i in range(10): a_.scatter(i, 0, s=300, color=ACCENT if i < k else "white", edgecolors=ACCENT, linewidths=2)
        a_.axis("off"); a_.set_title(f"{lab}: of 10 alerts, about {k} were real", loc="left", fontsize=9)
    plt.tight_layout(); plt.show()

**✏️ Your answer (Part 7):** (a) Why does a red dot at every alert make Ravi's team **more** likely to miss the frozen H2 sensor? Use the words *salience* and *inattentional blindness*. (b) Rewrite vendor statement 6 using frequency framing.

**(a)** When every alert is bright red, all alerts compete for **salience** and the genuinely important pattern no longer stands out. Technicians can become habituated to the red marks and experience **inattentional blindness**, overlooking the frozen H2 sensor because attention is captured by the repeated alerts.

**(b)** “Among readings with scores of at least 0.90, about the number out of 10 shown in the frequency chart were confirmed faults; among readings scoring 0.50–0.70, use the corresponding number shown in the second row.” This frequency framing reports observed outcomes and does not pretend that a score of 0.92 is a calibrated 92% probability.

---
## Part 8 · Standard deviation vs standard error (vendor statement 8)

The vendor judged a new FZ-B1 reading of **−15.9 °C at 14:00** against the 14:00 baseline.

In [ ]:
base = df[(df.freezer_id == "FZ-B1") & (df.hour == 14) & (df.confirmed_fault == 0)]["temp_c"]
mu, sd, n = base.mean(), base.std(), len(base)
new_reading = -15.9

se = sd / np.sqrt(n)
z_sd = (new_reading - mu) / sd
z_se = (new_reading - mu) / se

if ready(se, z_sd, z_se):
    print(f"baseline: mean {mu:.2f} °C, SD {sd:.2f}, n = {n}, SE = {se:.3f}")
    print(f"z using SD = {z_sd:.1f} | z using SE = {z_se:.1f}")

correct_z = "SD"

**✏️ Your answer (Part 8):** In one sentence each, say what the SD and the SE describe, and rewrite vendor statement 8.

**SD** describes the spread of individual normal temperature readings around their baseline mean; **SE** describes the uncertainty or precision of the estimated baseline mean. Vendor statement 8 is wrong because a single new reading must be compared with the distribution of individual readings using SD, not the much smaller SE of the mean, so “15 standard errors” does not establish a definite fault.

---
## Part 9 · Triage the vendor report

For each statement decide: **Accept** (correct as stated), **Correct** (useful, but the wording or number must change) or **Reject** (not supported). Then measure the report.

In [ ]:
decisions = {
    1: "Correct",  # accuracy is misleading under class imbalance; report precision and recall
    2: "Reject",   # 2 am false alarms are not evidence that most true faults occur then
    3: "Correct",  # the z-score rule can miss an extreme value in a short window
    4: "Reject",   # flat sensor readings may indicate a collective/sensor failure
    5: "Reject",   # abnormality depends on freezer and hour context
    6: "Correct",  # score is not calibrated probability
    7: "Correct",  # threshold should reflect costs and missed-fault patterns
    8: "Reject",   # SD, not SE, judges a single new observation
}

if all(v in ("Accept", "Correct", "Reject") for v in decisions.values()):
    accepted = sum(v == "Accept" for v in decisions.values())
    print(f"Statements accepted as written: {accepted}/{len(decisions)} ({accepted / len(decisions):.0%})")
    print(pd.Series(decisions).value_counts().to_string())
else:
    print("⚠ Fill in all eight decisions with 'Accept', 'Correct' or 'Reject'.")

**✏️ Your answer (Part 9):** Sketch (in words or a quick drawing) the **anomaly review view** you would give Ravi's team instead of the vendor's red-dot dashboard. Name at least four parts: main view, context, queue, verification tools, evaluation.

I would give the team a **time-series main view** for each freezer, with temperature on the vertical axis and time on the horizontal axis. Add a **context band** showing that freezer's hour-of-day median and normal spread, and use graded colour/size for score strength rather than painting every alert red. A **prioritised queue** should list freezer, timestamp, score, anomaly type and reason for review; **verification tools** should show nearby readings, raw sensor values and maintenance-log history, with controls to confirm, dismiss or escalate an alert. Finally, an **evaluation panel** should report precision, recall, false alarms by hour and missed faults by type, with sample sizes and the selected threshold/cost assumptions.

---
## Reflection

Answer in 2–4 sentences each. There are no right-or-wrong answers here; we are looking for honest thinking about *your own* process.

**1. Automation bias.** Before you checked anything, which vendor statement did you find most convincing? Was it the number, the confident tone, or the red dots? Would you have caught the problem without the maintenance log?

**1.** The 97.9% accuracy statement initially sounded convincing because it used a precise percentage and implied that the product had already been independently validated. I would not have trusted it without checking the maintenance log, because a high accuracy can hide missed faults when normal readings greatly outnumber faulty ones.

**2.** The cold-chain lead should recommend the operational threshold, but the decision should be owned jointly by operations, maintenance and the person accountable for stock loss. The dashboard should show precision, recall, false-call-out volume, missed-fault cost, threshold alternatives and the fault types that remain undetected.

**3.** If faults that were never inspected are labelled as normal, the measured precision and recall can both be misleading: some apparent false alarms may be real faults, and some missed faults remain invisible. The workflow should sample and inspect a portion of unalerted readings, record technician outcomes consistently and allow labels to be corrected after follow-up.

**4.** I would replace the uniform red alert with a graded encoding and show each reading against its freezer- and hour-specific baseline. An alert should open the supporting time window and maintenance history, with clear confirm/dismiss controls, so technicians can verify the evidence rather than automatically obeying the alert.

**2. Who sets the threshold?** The threshold decides how many night call-outs happen and how many freezers spoil. Who at FreshKart should own that choice, and what information should the dashboard show them?

**1.** The 97.9% accuracy statement initially sounded convincing because it used a precise percentage and implied that the product had already been independently validated. I would not have trusted it without checking the maintenance log, because a high accuracy can hide missed faults when normal readings greatly outnumber faulty ones.

**2.** The cold-chain lead should recommend the operational threshold, but the decision should be owned jointly by operations, maintenance and the person accountable for stock loss. The dashboard should show precision, recall, false-call-out volume, missed-fault cost, threshold alternatives and the fault types that remain undetected.

**3.** If faults that were never inspected are labelled as normal, the measured precision and recall can both be misleading: some apparent false alarms may be real faults, and some missed faults remain invisible. The workflow should sample and inspect a portion of unalerted readings, record technician outcomes consistently and allow labels to be corrected after follow-up.

**4.** I would replace the uniform red alert with a graded encoding and show each reading against its freezer- and hour-specific baseline. An alert should open the supporting time window and maintenance history, with clear confirm/dismiss controls, so technicians can verify the evidence rather than automatically obeying the alert.

**3. Missing ground truth.** You had a maintenance log. In real life, faults nobody inspected never get a label. How would that change your precision and recall numbers, and how could the review workflow reduce the problem?

**1.** The 97.9% accuracy statement initially sounded convincing because it used a precise percentage and implied that the product had already been independently validated. I would not have trusted it without checking the maintenance log, because a high accuracy can hide missed faults when normal readings greatly outnumber faulty ones.

**2.** The cold-chain lead should recommend the operational threshold, but the decision should be owned jointly by operations, maintenance and the person accountable for stock loss. The dashboard should show precision, recall, false-call-out volume, missed-fault cost, threshold alternatives and the fault types that remain undetected.

**3.** If faults that were never inspected are labelled as normal, the measured precision and recall can both be misleading: some apparent false alarms may be real faults, and some missed faults remain invisible. The workflow should sample and inspect a portion of unalerted readings, record technician outcomes consistently and allow labels to be corrected after follow-up.

**4.** I would replace the uniform red alert with a graded encoding and show each reading against its freezer- and hour-specific baseline. An alert should open the supporting time window and maintenance history, with clear confirm/dismiss controls, so technicians can verify the evidence rather than automatically obeying the alert.

**4. Designing against over-trust.** Name one design change (encoding, interaction, wording) that would make technicians *check* an alert rather than simply obey or ignore it.

**1.** The 97.9% accuracy statement initially sounded convincing because it used a precise percentage and implied that the product had already been independently validated. I would not have trusted it without checking the maintenance log, because a high accuracy can hide missed faults when normal readings greatly outnumber faulty ones.

**2.** The cold-chain lead should recommend the operational threshold, but the decision should be owned jointly by operations, maintenance and the person accountable for stock loss. The dashboard should show precision, recall, false-call-out volume, missed-fault cost, threshold alternatives and the fault types that remain undetected.

**3.** If faults that were never inspected are labelled as normal, the measured precision and recall can both be misleading: some apparent false alarms may be real faults, and some missed faults remain invisible. The workflow should sample and inspect a portion of unalerted readings, record technician outcomes consistently and allow labels to be corrected after follow-up.

**4.** I would replace the uniform red alert with a graded encoding and show each reading against its freezer- and hour-specific baseline. An alert should open the supporting time window and maintenance history, with clear confirm/dismiss controls, so technicians can verify the evidence rather than automatically obeying the alert.